# PIC-GD FMA-Small Colab Notebook

This notebook sets up the `phase-induced-coherence-gated-gradient-descent` repo in Google Colab and runs the FMA-small experiment path.

It includes:
- repo setup
- minimal dependency install
- FMA-small download and unpack
- a smoke test
- a full training command


## 1. Clone The Repo

If you want a non-default branch, replace `main` below with the branch name before running the cell.

In [ ]:
REPO_URL = "https://github.com/jzgdev/phase-induced-coherence-gated-gradient-descent.git"
REPO_DIR = "/content/phase-induced-coherence-gated-gradient-descent"
BRANCH = "main"

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!git checkout "$BRANCH"


## 2. Install Runtime Dependencies

Colab already ships with PyTorch in most GPU runtimes. This cell installs only the extra packages the repo relies on for audio loading and testing.

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q soundfile torchaudio

## 3. Download FMA-Small

This pulls the official FMA-small audio subset and metadata. The expected final layout is:

```text
/content/data/fma/
├── fma_small/
└── fma_metadata/
```

In [ ]:
import os

DATA_ROOT = "/content/data/fma"
os.makedirs(DATA_ROOT, exist_ok=True)
%cd $DATA_ROOT

!wget -nc https://os.unil.cloud.switch.ch/fma/fma_small.zip
!wget -nc https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
!unzip -n fma_small.zip
!unzip -n fma_metadata.zip


## 4. Smoke Test

Run a short sanity check first so you can confirm the dataset, decoding, and model path work before committing to a long training job.

In [ ]:
%cd /content/phase-induced-coherence-gated-gradient-descent

!python phase_coherence_test.py \
  --dataset fma_small \
  --fma_root /content/data/fma/fma_small \
  --fma_metadata_root /content/data/fma/fma_metadata \
  --sample_rate 48000 \
  --batch_size 8 \
  --num_workers 2 \
  --num_runs 1 \
  --epochs 1 \
  --train_samples_per_epoch 128 \
  --val_samples_per_epoch 64

## 5. Full FMA-Small Run

Increase workers or batch size depending on the Colab runtime you get. If you want checkpoints, add `--save_checkpoints`.

In [ ]:
%cd /content/phase-induced-coherence-gated-gradient-descent

!python phase_coherence_test.py \
  --dataset fma_small \
  --fma_root /content/data/fma/fma_small \
  --fma_metadata_root /content/data/fma/fma_metadata \
  --sample_rate 48000 \
  --batch_size 16 \
  --num_workers 4 \
  --variants baseline,complex,align,gate_only,full \
  --analysis_variants baseline,full \
  --generate_plots

## 6. Inspect Outputs

Results are written under `results/<dataset>/<eval_protocol>/seed_<seed>/`.

In [ ]:
!find /content/phase-induced-coherence-gated-gradient-descent/results -maxdepth 4 -type f | sort | tail -n 50